##**Fine-tuning**

In [1]:
from datasets import load_dataset

dataset = load_dataset('stanfordnlp/imdb')
print(dataset)

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [2]:
print(dataset['train'][:3])

{'text': ['I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far b

In [3]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
  return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

small_train = dataset["train"].shuffle(seed=42).select(range(500))
small_test = dataset["test"].shuffle(seed=42).select(range(250))

small_train = small_train.map(tokenize_fn, batched=True)
small_test = small_test.map(tokenize_fn, batched=True)

print(small_train[0].keys())

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])


In [4]:
print(len(small_train[0]['input_ids']))

256


In [5]:
import sys
!{sys.executable} -m pip install --upgrade torchao

from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q_lin', 'v_lin']
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 58.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


In [6]:
print(model)

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): DistilBertSelfAttention(
                (q_lin): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=False)
                  )
      

In [7]:
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00


In [8]:
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate

accuracy_metric = evaluate.load('accuracy')
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  return accuracy_metric.compute(predictions=predictions, references=labels)


training_args = TrainingArguments(
    output_dir="./lora_imdb",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    logging_steps=10,
    learning_rate=1e-3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics
)

trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy
1,0.420519,0.448177,0.816000
2,0.310250,0.336229,0.856000
3,0.205289,0.348878,0.860000


TrainOutput(global_step=96, training_loss=0.39366724466284114, metrics={'train_runtime': 32.3704, 'train_samples_per_second': 46.339, 'train_steps_per_second': 2.966, 'total_flos': 101054555136000.0, 'train_loss': 0.39366724466284114, 'epoch': 3.0})

In [9]:
import torch
text = "The acting was so beautifully over-the-top that I honestly couldn't tell if I was watching a total disaster or a misunderstood stroke of absolute genius."

def predict_sentiment(text, model, tokenizer):
  inputs = tokenizer(text, return_tensors='pt', truncation=True, padding='max_length', max_length=256)
  inputs = {k: v.to(model.device) for k, v in inputs.items()}

  model.eval()
  with torch.no_grad():
    outputs = model(**inputs)

  probs = torch.softmax(outputs.logits, dim=-1)
  pred = probs.argmax(dim=-1).item()
  label = "POSITIVE" if pred==1 else "NEGATIVE"
  confidence = probs[0][pred].item()

  return label, confidence

label, confidence = predict_sentiment(text,model,tokenizer)
print(f"label: {label}, confidence: {confidence}")

label: NEGATIVE, confidence: 0.5030748248100281


In [10]:
model.save_pretrained('./lora_imdb_small_500')

##**Full Dataset fine-tuning**

In [11]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType

# init
model_name = "distilbert-base-uncased"
dataset = load_dataset('stanfordnlp/imdb')

train = dataset['train'].shuffle(seed=42)
test = dataset['test'].shuffle(seed=42)

# tokenize data
tokenizer = AutoTokenizer.from_pretrained(model_name)
def tokenizer_fn(examples):
  return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=256)

train = train.map(tokenizer_fn, batched=True)
test = test.map(tokenizer_fn, batched=True)


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [12]:
import sys
!{sys.executable} -m pip install --upgrade torchao

In [13]:
# Fine-tuning with lora
base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    target_modules=['q_lin','v_lin'],
    r=8, #default
    lora_alpha=16, #usually better to be the double of r, here r=8 then put 16
    lora_dropout=0.1 #usually 0.1 or 0.05 for normal training and 0.0 for massive datasets
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters() # good to check the parameters

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


In [14]:
!pip install -q evaluate

In [15]:
# Train the model using the dataset
from transformers import Trainer, TrainingArguments
import numpy as np
import evaluate

accuracy_metric = evaluate.load('accuracy')
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  return accuracy_metric.compute(predictions=predictions, references=labels)

training_arguments = TrainingArguments(
    output_dir="./lora_imdb", # the directory that the model will save its logs, checkpoints, and outputs to
    learning_rate=1e-3, # learning rate for the optimizer defaults to 5e-5
    per_device_train_batch_size=16, # the batch size per GPU/CPU for training default is 8
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch", # when to run the evaluation? no, steps, or epoch
    logging_steps=10, # number of update steps between two logs default is 500
)

trainer = Trainer(
    args=training_arguments,
    model=model, # the model to train
    train_dataset=train, # the dataset used to train
    eval_dataset=test, # the dataset used to evaluate the model performance
    compute_metrics=compute_metrics # used to evaluate
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.398416,0.283668,0.895480
2,0.323694,0.225686,0.909960
3,0.157611,0.244289,0.912440


TrainOutput(global_step=4689, training_loss=0.23798360682723618, metrics={'train_runtime': 1808.7215, 'train_samples_per_second': 41.466, 'train_steps_per_second': 2.592, 'total_flos': 5052727756800000.0, 'train_loss': 0.23798360682723618, 'epoch': 3.0})

In [16]:
# Test accuracy of the trained model
import torch

def predict_sentiment(text, tokenizer, model):
  inputs = tokenizer(text, return_tensors='pt', truncation=True, padding="max_length", max_length=256)
  inputs = {k: v.to(model.device) for k, v in inputs.items()}

  model.eval()
  with torch.no_grad():
    outputs = model(**inputs)

  probs = torch.softmax(outputs.logits, dim=-1)
  pred = probs.argmax(dim=-1).item()
  label = "POSITIVE" if pred == 1 else "NEGATIVE"
  confidence = probs[0][pred].item()

  return label, confidence

label, confidence = predict_sentiment(
    text="Cats (2019) is a chaotic, unsettling misfire that turns a beloved stage musical into a nightmarish digital fever dream.",
    tokenizer=tokenizer,
    model=model
)

print(f"label: {label}, confidence: {confidence}")


label: POSITIVE, confidence: 0.9118684530258179


Tried:
1. The acting was so beautifully over-the-top that I honestly couldn't tell if I was watching a total disaster or a misunderstood stroke of absolute genius. (label: POSITIVE, confidence: 0.9797607660293579)

2. Truly a masterclass in cinematic filmmaking, provided your goal is to make the audience root for the literal meteor to wipe out the entire cast. (label: POSITIVE, confidence: 0.9899117350578308)

3. I absolutely loved how the plot holes were big enough to drive a truck through—it really gave my brain a much-needed rest from thinking. (label: POSITIVE, confidence: 0.9120398759841919)

In [17]:
model.save_pretrained("./lora_imdb_adapter")